# Notebook Overview — Build Feature Vectors

## Purpose

This notebook combines the **gradient**, **spatial**, and **frequency-domain** feature sets into unified **Digital Image Processing (DIP) feature vectors** for both the **training** and **test** datasets.

It ensures that all feature sets are aligned, validated, and merged into a single structured dataset suitable for downstream normalization and classification.

---

## Inputs

Required inputs:

* Gradient feature CSVs:
  * `metadata/features/<train_gradient_features_filename>`
  * `metadata/features/<test_gradient_features_filename>`

* Spatial feature CSVs:
  * `metadata/features/<train_spatial_features_filename>`
  * `metadata/features/<test_spatial_features_filename>`

* Frequency feature CSVs:
  * `metadata/features/<train_frequency_features_filename>`
  * `metadata/features/<test_frequency_features_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Training and test datasets are processed independently
* Feature CSV files are loaded for gradient, spatial, and frequency domains
* Row counts and metadata alignment are validated across all feature sets
* Required columns and subset labels are verified
* Feature column groups are explicitly defined
* Feature sets are concatenated into unified feature vector tables
* Final feature vectors are validated for structure, completeness, and consistency
* Output datasets are saved to CSV files
* Processing is deterministic and reproducible

---

## Outputs

**Training Feature Vector CSV**  
`metadata/vectors/<train_feature_vectors_filename>`

**Test Feature Vector CSV**  
`metadata/vectors/<test_feature_vectors_filename>`

Each output file contains:

* Metadata columns:
  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* Feature columns (25 total):

  **Gradient Features (8):**
  * Mean Gradient  
  * Std Gradient  
  * Max Gradient  
  * Gradient Entropy  
  * Edge Density  
  * Orientation Mean  
  * Orientation Std  
  * Orientation Entropy  

  **Spatial Features (9):**
  * Global Entropy  
  * Local Entropy Mean  
  * Local Entropy Std  
  * Intensity Mean  
  * Intensity Std  
  * Laplacian Variance  
  * Patch Variance Mean  
  * Patch Variance Std  
  * Noise Residual Energy  

  **Frequency-Domain Features (8):**
  * Low Frequency Energy Ratio  
  * High Frequency Energy Ratio  
  * Radial Mean  
  * Radial Std  
  * Radial Entropy  
  * Spectral Centroid  
  * Spectral Bandwidth  
  * Log Spectrum Std  

**Expected Behavior**

* Each image is represented by a **25-dimensional feature vector**  
* Output row counts match the corresponding input datasets  
* Metadata remains fully aligned across all feature sets  
* No missing values are present in the final feature vectors  
* Output is ready for normalization and classifier input preparation  

---
---

### 🔷 Step 1 — Startup: Environment and Verification

- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input feature CSV files for gradient, spatial, and frequency features
- Define output paths for combined feature vectors
- Create output directory if it does not exist
- Verify that all required feature CSV files are present
- Optionally display input and output file paths when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Startup — Environment and Verification
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_GRADIENT_FEATURES_PATH,
    TRAIN_SPATIAL_FEATURES_PATH,
    TRAIN_FREQUENCY_FEATURES_PATH,
    TEST_GRADIENT_FEATURES_PATH,
    TEST_SPATIAL_FEATURES_PATH,
    TEST_FREQUENCY_FEATURES_PATH,
    TRAIN_FEATURE_VECTOR_PATH,
    TEST_FEATURE_VECTOR_PATH,
    TRAIN_IMAGES,
    TEST_IMAGES,
    TRAIN_SUBSET,
    TEST_SUBSET,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
INPUT_FILES = [
    Path(TRAIN_GRADIENT_FEATURES_PATH),
    Path(TRAIN_SPATIAL_FEATURES_PATH),
    Path(TRAIN_FREQUENCY_FEATURES_PATH),
    Path(TEST_GRADIENT_FEATURES_PATH),
    Path(TEST_SPATIAL_FEATURES_PATH),
    Path(TEST_FREQUENCY_FEATURES_PATH),
]

TRAIN_OUTPUT_FILE = Path(TRAIN_FEATURE_VECTOR_PATH)
TEST_OUTPUT_FILE = Path(TEST_FEATURE_VECTOR_PATH)

OUTPUT_DIR = TRAIN_OUTPUT_FILE.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required feature input files
# -------------------------------------------------
print("Verifying required feature CSV files...\n")

missing_files = [str(f) for f in INPUT_FILES if not f.exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing required feature files:\n{missing_files}"
    )

print("All required feature CSV files are present.")

# -------------------------------------------------
# Optional verbose display of input/output paths
# -------------------------------------------------
if VERBOSE:
    print("\nInput files:")
    for f in INPUT_FILES:
        print(f"  - {f}")

    print(f"\nOutput files:")
    print(f"  - Train: {TRAIN_OUTPUT_FILE}")
    print(f"  - Test : {TEST_OUTPUT_FILE}")

print("\nStartup complete.")



### 🔷 Step 2 — Define Expected Sizes and Display Paths

- Define expected row counts for train and test subsets
- Display input feature files for gradient, spatial, and frequency features
- Display output paths for combined feature vectors
- Display expected dataset sizes for validation

---

In [ ]:
# ============================================
# Step 2: Define Expected Sizes and Display Paths
# ============================================

# -------------------------------------------------
# Define expected row counts for each subset
# -------------------------------------------------
EXPECTED_ROWS = {
    TRAIN_SUBSET: TRAIN_IMAGES,
    TEST_SUBSET: TEST_IMAGES,
}

# -------------------------------------------------
# Display training configuration
# -------------------------------------------------
print("Training input files:")
print(f"  GRADIENT      = {TRAIN_GRADIENT_FEATURES_PATH}")
print(f"  SPATIAL       = {TRAIN_SPATIAL_FEATURES_PATH}")
print(f"  FREQUENCY     = {TRAIN_FREQUENCY_FEATURES_PATH}")
print(f"  OUTPUT        = {TRAIN_FEATURE_VECTOR_PATH}")
print(f"  EXPECTED ROWS = {EXPECTED_ROWS[TRAIN_SUBSET]}")

# -------------------------------------------------
# Display test configuration
# -------------------------------------------------
print("\nTest input files:")
print(f"  GRADIENT      = {TEST_GRADIENT_FEATURES_PATH}")
print(f"  SPATIAL       = {TEST_SPATIAL_FEATURES_PATH}")
print(f"  FREQUENCY     = {TEST_FREQUENCY_FEATURES_PATH}")
print(f"  OUTPUT        = {TEST_FEATURE_VECTOR_PATH}")
print(f"  EXPECTED ROWS = {EXPECTED_ROWS[TEST_SUBSET]}")

print("\nPath and expected-size configuration complete.")



### 🔷 Step 3 — Load Feature CSVs

- Define input paths for gradient, spatial, and frequency feature CSV files
- Verify that all required input files exist
- Load feature CSV files for both training and test subsets
- Display shapes of loaded datasets for validation
- Display expected row counts for comparison

---

In [ ]:
# ============================================
# Step 3: Load Feature CSVs
# ============================================

# -------------------------------------------------
# Define input paths for training and test feature CSVs
# -------------------------------------------------
train_paths = [
    TRAIN_GRADIENT_FEATURES_PATH,
    TRAIN_SPATIAL_FEATURES_PATH,
    TRAIN_FREQUENCY_FEATURES_PATH,
]

test_paths = [
    TEST_GRADIENT_FEATURES_PATH,
    TEST_SPATIAL_FEATURES_PATH,
    TEST_FREQUENCY_FEATURES_PATH,
]

# -------------------------------------------------
# Verify input files exist
# -------------------------------------------------
for path in train_paths + test_paths:
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing required file: {path}")

# -------------------------------------------------
# Load training feature CSVs
# -------------------------------------------------
df_grad_train = pd.read_csv(TRAIN_GRADIENT_FEATURES_PATH)
df_spat_train = pd.read_csv(TRAIN_SPATIAL_FEATURES_PATH)
df_freq_train = pd.read_csv(TRAIN_FREQUENCY_FEATURES_PATH)

# -------------------------------------------------
# Load test feature CSVs
# -------------------------------------------------
df_grad_test = pd.read_csv(TEST_GRADIENT_FEATURES_PATH)
df_spat_test = pd.read_csv(TEST_SPATIAL_FEATURES_PATH)
df_freq_test = pd.read_csv(TEST_FREQUENCY_FEATURES_PATH)

# -------------------------------------------------
# Display summary of loaded datasets
# -------------------------------------------------
print("Training CSV shapes:")
print("  Gradient : ", df_grad_train.shape)
print("  Spatial  : ", df_spat_train.shape)
print("  Frequency: ", df_freq_train.shape)

print("\nTest CSV shapes:")
print("  Gradient : ", df_grad_test.shape)
print("  Spatial  : ", df_spat_test.shape)
print("  Frequency: ", df_freq_test.shape)

# -------------------------------------------------
# Display expected dataset sizes
# -------------------------------------------------
print("\nExpected rows:")
print("  Train:", EXPECTED_ROWS[TRAIN_SUBSET])
print("  Test :", EXPECTED_ROWS[TEST_SUBSET])



### 🔷 Step 4 — Validate Row Counts and Alignment

- Define metadata columns required for alignment checks
- Validate row counts for gradient, spatial, and frequency feature CSVs
- Verify that metadata columns match across all feature CSVs
- Perform validation separately for training and test subsets
- Confirm that all input feature files are consistent and aligned
- Optionally display validation results when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 4: Validate Row Counts and Alignment
# ============================================

# -------------------------------------------------
# Define metadata columns for alignment checks
# -------------------------------------------------
metadata_columns = ["filename", "source_dataset", "class_label", "subset"]

# -------------------------------------------------
# Validate row counts and metadata alignment
# -------------------------------------------------
def validate_subset_alignment(df_grad, df_spat, df_freq, expected_rows, subset_name):
    # -------------------------------------------------
    # Validate row counts
    # -------------------------------------------------
    if len(df_grad) != expected_rows:
        raise ValueError(
            f"{subset_name} gradient CSV row count mismatch: expected {expected_rows}, got {len(df_grad)}"
        )

    if len(df_spat) != expected_rows:
        raise ValueError(
            f"{subset_name} spatial CSV row count mismatch: expected {expected_rows}, got {len(df_spat)}"
        )

    if len(df_freq) != expected_rows:
        raise ValueError(
            f"{subset_name} frequency CSV row count mismatch: expected {expected_rows}, got {len(df_freq)}"
        )

    # -------------------------------------------------
    # Validate metadata alignment across CSVs
    # -------------------------------------------------
    for col in metadata_columns:
        if not df_grad[col].equals(df_spat[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and spatial CSVs in column: {col}"
            )
        if not df_grad[col].equals(df_freq[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and frequency CSVs in column: {col}"
            )

    # -------------------------------------------------
    # Optional verbose confirmation
    # -------------------------------------------------
    if VERBOSE:
        print(f"PASS: {subset_name} input CSVs have the expected row count")
        print(f"PASS: {subset_name} metadata columns align across all three CSVs")


# -------------------------------------------------
# Validate training subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nRow count and alignment validation complete.")



### 🔷 Step 5 — Define Metadata and Feature Columns

- Define metadata columns used across all feature sets
- Define gradient feature column names
- Define spatial feature column names
- Define frequency-domain feature column names
- Display the number of features in each group
- Display the total number of DIP features

---

In [ ]:
# ============================================
# Step 5: Define Metadata and Feature Columns
# ============================================

# -------------------------------------------------
# Define metadata columns
# -------------------------------------------------
metadata_cols = metadata_columns

# -------------------------------------------------
# Define gradient feature columns
# -------------------------------------------------
gradient_feature_cols = [
    "Mean Gradient",
    "Std Gradient",
    "Max Gradient",
    "Gradient Entropy",
    "Edge Density",
    "Orientation Mean",
    "Orientation Std",
    "Orientation Entropy"
]

# -------------------------------------------------
# Define spatial feature columns
# -------------------------------------------------
spatial_feature_cols = [
    "Global Entropy",
    "Local Entropy Mean",
    "Local Entropy Std",
    "Intensity Mean",
    "Intensity Std",
    "Laplacian Variance",
    "Patch Variance Mean",
    "Patch Variance Std",
    "Noise Residual Energy"
]

# -------------------------------------------------
# Define frequency-domain feature columns
# -------------------------------------------------
frequency_feature_cols = [
    "Low Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std"
]

# -------------------------------------------------
# Display feature group sizes
# -------------------------------------------------
print("Metadata columns: ", len(metadata_cols))
print("Gradient features:", len(gradient_feature_cols))
print("Spatial features: ", len(spatial_feature_cols))
print("Frequency features:", len(frequency_feature_cols))

# -------------------------------------------------
# Display total number of DIP features
# -------------------------------------------------
print("Total DIP features:",
      len(gradient_feature_cols)
    + len(spatial_feature_cols)
    + len(frequency_feature_cols))



### 🔷 Step 6 — Verify Required Columns Exist

- Define a helper function to validate required columns in a DataFrame
- Verify that gradient, spatial, and frequency CSVs contain expected metadata and feature columns
- Perform validation separately for training and test subsets
- Confirm that all required columns are present before merging
- Optionally display validation results when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 6: Verify Required Columns Exist
# ============================================

# -------------------------------------------------
# Verify required columns exist in a DataFrame
# -------------------------------------------------
def check_columns(df, required_cols, df_name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")
    if VERBOSE:
        print(f"{df_name}: all required columns present")

# -------------------------------------------------
# Validate training feature CSVs
# -------------------------------------------------
check_columns(df_grad_train, metadata_cols + gradient_feature_cols, "Train Gradient CSV")
check_columns(df_spat_train, metadata_cols + spatial_feature_cols, "Train Spatial CSV")
check_columns(df_freq_train, metadata_cols + frequency_feature_cols, "Train Frequency CSV")

# -------------------------------------------------
# Validate test feature CSVs
# -------------------------------------------------
check_columns(df_grad_test, metadata_cols + gradient_feature_cols, "Test Gradient CSV")
check_columns(df_spat_test, metadata_cols + spatial_feature_cols, "Test Spatial CSV")
check_columns(df_freq_test, metadata_cols + frequency_feature_cols, "Test Frequency CSV")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nColumn verification complete.")



### 🔷 Step 7 — Verify Metadata Alignment Across CSVs

- Define a helper function to compare metadata columns across feature CSVs
- Verify that metadata columns match between gradient, spatial, and frequency datasets
- Perform alignment checks separately for training and test subsets
- Detect and raise errors if any mismatch is found
- Optionally display detailed comparison results when `VERBOSE=True`
- Confirm that all metadata is fully aligned before merging

---

In [ ]:
# ============================================
# Step 7: Verify Metadata Alignment Across CSVs
# ============================================

# -------------------------------------------------
# Verify metadata alignment across feature CSVs
# -------------------------------------------------
def check_metadata_alignment(df_grad, df_spat, df_freq, subset_name):
    for col in metadata_cols:
        same_grad_spat = df_grad[col].equals(df_spat[col])
        same_grad_freq = df_grad[col].equals(df_freq[col])

        # -------------------------------------------------
        # Optional verbose comparison output
        # -------------------------------------------------
        if VERBOSE:
            print(f"{subset_name} | {col}:")
            print(f"  Gradient vs Spatial:   {same_grad_spat}")
            print(f"  Gradient vs Frequency: {same_grad_freq}")

        # -------------------------------------------------
        # Raise error if any mismatch is detected
        # -------------------------------------------------
        if not same_grad_spat or not same_grad_freq:
            raise ValueError(f"{subset_name}: mismatch detected in metadata column: {col}")

    print(f"PASS: {subset_name} metadata columns align across CSVs")


# -------------------------------------------------
# Validate training subset metadata alignment
# -------------------------------------------------
check_metadata_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test subset metadata alignment
# -------------------------------------------------
check_metadata_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nMetadata alignment verification complete.")


### 🔷 Step 8 — Verify Subset Labels Match Expected Subsets

- Define a helper function to validate subset labels across feature CSVs
- Extract and compare unique subset values for gradient, spatial, and frequency datasets
- Verify that each dataset contains only the expected subset label (train or test)
- Perform validation separately for training and test subsets
- Raise errors if any subset mismatch is detected
- Optionally display subset values when `VERBOSE=True`
- Confirm that subset labels are correct before merging feature vectors

---

In [ ]:
# ============================================
# Step 8: Verify Subset Labels Match Expected Subsets
# ============================================

# -------------------------------------------------
# Verify subset labels are consistent and correct
# -------------------------------------------------
def check_subset_labels(df_grad, df_spat, df_freq, expected_subset_value):
    # Extract unique subset values from each DataFrame
    unique_subsets_grad = sorted(df_grad["subset"].dropna().unique())
    unique_subsets_spat = sorted(df_spat["subset"].dropna().unique())
    unique_subsets_freq = sorted(df_freq["subset"].dropna().unique())

    # -------------------------------------------------
    # Optional verbose display of subset values
    # -------------------------------------------------
    if VERBOSE:
        print(f"{expected_subset_value} | Gradient subset values: ", unique_subsets_grad)
        print(f"{expected_subset_value} | Spatial subset values:  ", unique_subsets_spat)
        print(f"{expected_subset_value} | Frequency subset values:", unique_subsets_freq)

    # -------------------------------------------------
    # Validate subset labels for each feature set
    # -------------------------------------------------
    if unique_subsets_grad != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} gradient CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_grad}"
        )

    if unique_subsets_spat != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} spatial CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_spat}"
        )

    if unique_subsets_freq != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} frequency CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_freq}"
        )

    print(f"PASS: subset labels match {expected_subset_value}")


# -------------------------------------------------
# Validate training subset labels
# -------------------------------------------------
check_subset_labels(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test subset labels
# -------------------------------------------------
check_subset_labels(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nSubset label verification complete.")



### 🔷 Step 9 — Build Final Feature Vector DataFrames

- Combine metadata and feature columns into unified feature vector DataFrames
- Concatenate gradient, spatial, and frequency features with shared metadata
- Construct separate feature vector datasets for training and test subsets
- Verify resulting DataFrame shapes
- Optionally display sample rows for inspection when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 9: Build Final Feature Vector DataFrames
# ============================================

# -------------------------------------------------
# Build training feature vector DataFrame
# -------------------------------------------------
df_feature_vectors_train = pd.concat(
    [
        # Metadata (shared across all feature sets)
        df_grad_train[metadata_cols],

        # Feature groups
        df_grad_train[gradient_feature_cols],
        df_spat_train[spatial_feature_cols],
        df_freq_train[frequency_feature_cols]
    ],
    axis=1
)

# -------------------------------------------------
# Build test feature vector DataFrame
# -------------------------------------------------
df_feature_vectors_test = pd.concat(
    [
        # Metadata (shared across all feature sets)
        df_grad_test[metadata_cols],

        # Feature groups
        df_grad_test[gradient_feature_cols],
        df_spat_test[spatial_feature_cols],
        df_freq_test[frequency_feature_cols]
    ],
    axis=1
)

# -------------------------------------------------
# Display summary of resulting feature vectors
# -------------------------------------------------
print("Training feature vector shape:", df_feature_vectors_train.shape)
print("Test feature vector shape:", df_feature_vectors_test.shape)

# -------------------------------------------------
# Optional verbose display of sample rows
# -------------------------------------------------
if VERBOSE:
    print("\nSample training rows:")
    display(df_feature_vectors_train.head(3))

    print("\nSample test rows:")
    display(df_feature_vectors_test.head(3))



### 🔷 Step 10 — Validate Final Feature Vector Tables

- Compute expected total number of feature columns
- Validate that feature vector tables contain the correct number of features
- Verify row counts match expected dataset sizes
- Check for missing values in the final feature vectors
- Optionally display class, source, and subset distributions when `VERBOSE=True`
- Confirm that both training and test feature vector tables are valid and complete

---

In [ ]:
# ============================================
# Step 10: Validate Final Feature Vector Tables
# ============================================

# -------------------------------------------------
# Compute expected number of feature columns
# -------------------------------------------------
expected_feature_count = (
    len(gradient_feature_cols) +
    len(spatial_feature_cols) +
    len(frequency_feature_cols)
)

# -------------------------------------------------
# Validate feature vector table structure and contents
# -------------------------------------------------
def validate_feature_vector_table(df_feature_vectors, expected_rows, subset_name):
    # -------------------------------------------------
    # Validate feature column count
    # -------------------------------------------------
    actual_feature_count = len(df_feature_vectors.columns) - len(metadata_cols)

    print(f"{subset_name} | Expected feature count: {expected_feature_count}")
    print(f"{subset_name} | Actual feature count:   {actual_feature_count}")

    if actual_feature_count != expected_feature_count:
        raise ValueError(f"{subset_name}: feature count mismatch")

    # -------------------------------------------------
    # Validate row count
    # -------------------------------------------------
    if len(df_feature_vectors) != expected_rows:
        raise ValueError(
            f"{subset_name}: final row count mismatch: "
            f"expected {expected_rows}, got {len(df_feature_vectors)}"
        )

    # -------------------------------------------------
    # Check for missing values
    # -------------------------------------------------
    missing_counts = df_feature_vectors.isnull().sum()
    missing_counts = missing_counts[missing_counts > 0]

    if len(missing_counts) > 0:
        raise ValueError(f"{subset_name}: missing values detected:\n{missing_counts}")

    # -------------------------------------------------
    # Optional verbose distribution summaries
    # -------------------------------------------------
    if VERBOSE:
        print(f"\n{subset_name} | Class counts:")
        print(df_feature_vectors["class_label"].value_counts())

        print(f"\n{subset_name} | Source counts:")
        print(df_feature_vectors["source_dataset"].value_counts())

        print(f"\n{subset_name} | Subset counts:")
        print(df_feature_vectors["subset"].value_counts())

    print(f"\nPASS: {subset_name} feature vector table validated")


# -------------------------------------------------
# Validate training feature vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test feature vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nFinal feature vector validation complete.")



### 🔷 Step 11 — Save Final Feature Vectors

- Create output directory if it does not exist
- Save training feature vector DataFrame to CSV
- Save test feature vector DataFrame to CSV
- Optionally display save locations and shapes when `VERBOSE=True`
- Verify that output files were created successfully
- Confirm that feature vectors are saved and ready for the next stage

---

In [ ]:
# ============================================
# Step 11: Save Final Feature Vectors
# ============================================

# -------------------------------------------------
# Ensure output directory exists
# -------------------------------------------------
TRAIN_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Save training feature vector DataFrame
# -------------------------------------------------
df_feature_vectors_train.to_csv(TRAIN_OUTPUT_FILE, index=False)

if VERBOSE:
    print(f"Saved: {TRAIN_OUTPUT_FILE}")
    print("Train shape:", df_feature_vectors_train.shape)

# -------------------------------------------------
# Save test feature vector DataFrame
# -------------------------------------------------
df_feature_vectors_test.to_csv(TEST_OUTPUT_FILE, index=False)

if VERBOSE:
    print(f"Saved: {TEST_OUTPUT_FILE}")
    print("Test shape:", df_feature_vectors_test.shape)

# -------------------------------------------------
# Verify output files were created
# -------------------------------------------------
if not TRAIN_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Train output file not created: {TRAIN_OUTPUT_FILE}")

if not TEST_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Test output file not created: {TEST_OUTPUT_FILE}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nFeature vectors saved and verified.")

